In [0]:
travel_journal_dataset = [
    {"table": "accounts"},
    {"table": "activity_code"},
    {"table": "activity_tag"},
    {"table": "bio"},
    {"table": "country_code"},
    {"table": "follows"},
    {"table": "google_maps_address"},
    {"table": "images"},
    {"table": "location_country_tag"},
    {"table": "plan_trip"},
    {"table": "post_bookmarks"},
    {"table": "post_likes"},
    {"table": "trip_posts"},
    {"table": "trip_stops"},
]

source = "abfss://travelsource@databricktraveljournal.dfs.core.windows.net"
bronze = "abfss://bronze@databricktraveljournal.dfs.core.windows.net"


def has_parquet(path):
    for item in dbutils.fs.ls(path):
        if item.isDir():
            if has_parquet(item.path):
                return True
        elif item.path.endswith(".parquet"):
            print("found:", item.path)
            return True
    return False


def bronze_write_stream(table):                       # table is now a parameter
    source_table            = f"{source}/{table}"
    bronze_schema_destination = f"{bronze}/_schema/{table}"

    if has_parquet(source_table):
        bronze_stream = (spark.readStream.format("cloudFiles")
            .option("cloudFiles.format", "parquet")
            .option("cloudFiles.schemaLocation", bronze_schema_destination)
            .load(source_table))

        query = (bronze_stream.writeStream.format("parquet")
            .outputMode("append")
            .option("checkpointLocation", f"{bronze}/_checkpoint/{table}")
            .option("path", f"{bronze}/{table}")
            .trigger(availableNow=True)
            .start())

        query.awaitTermination()                      # wait before next table
    else:
        print(f"No data exist in source table parquet folder {source_table}")


if __name__ == "__main__":
    for item in travel_journal_dataset:               # loop over the list
        table = item["table"]                         # pull out ONE name
        print(f"=== Ingesting {table} ===")
        bronze_write_stream(table)